# BPATMP - Training + Evaluate trên Colab Pro (A100)

Hệ khuyến nghị đồ thị không đồng nhất trên REES46, **có xử lý cold-start**: item embedding được nối đất bằng content (category + brand), ID-dropout + history-dropout lúc train, và báo cáo **cold/warm** mỗi epoch.

**Cách dùng:** Runtime -> Change runtime type -> **A100 GPU** -> **Runtime -> Run all**.

Notebook tự động: clone repo -> kiểm tra code có cold-start -> tải dataset (HF, 100.775 item, kèm cờ `user_is_cold`/`item_is_cold`) -> train (đánh giá **val** + cold/warm mỗi epoch, log W&B) -> đánh giá **test** (graph train+val) và xuất bảng warm vs cold.

> **Bắt buộc trước khi chạy:** code cold-start phải đã được **commit & push** lên GitHub (`heterogeneous-graph-recsys`, nhánh `main`). Cell 3 sẽ tự kiểm tra và dừng sớm nếu repo clone về chưa có cold-start.
>
> **W&B:** thêm API key vào Colab Secrets () tên `WANDB_API_KEY`. Repo private thì thêm `GITHUB_TOKEN`.

## 1. Kiểm tra GPU (phải là A100)

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

name, memory.total [MiB], driver_version
NVIDIA A100-SXM4-80GB, 81920 MiB, 580.82.07
torch 2.11.0+cu128 | CUDA True | NVIDIA A100-SXM4-80GB


## 2. Cài dependencies
Colab đã có sẵn torch+CUDA, pandas, pyarrow, scipy, pyyaml, numpy. Code chỉ cần thêm `torch_geometric` (không cần torch_scatter/sparse/cluster), `wandb`, `huggingface_hub`.

In [2]:
!pip -q install torch_geometric wandb huggingface_hub hf_transfer
import torch_geometric, wandb, huggingface_hub
print('torch_geometric', torch_geometric.__version__, '| wandb', wandb.__version__, '| hf_hub', huggingface_hub.__version__)

torch_geometric 2.8.0 | wandb 0.27.2 | hf_hub 1.18.0


## 3. Clone repo
Nếu repo **private**: thêm secret `GITHUB_TOKEN` rồi bỏ comment khối token bên dưới.

In [ ]:
%cd /content
REPO = 'https://github.com/nguyenmaiductrong/heterogeneous-graph-recsys.git'
BRANCH = 'main'
# --- Repo private? Bỏ comment 3 dòng sau ---
# from google.colab import userdata
# TOK = userdata.get('GITHUB_TOKEN')
# REPO = f'https://{TOK}@github.com/nguyenmaiductrong/heterogeneous-graph-recsys.git'
import os
if not os.path.isdir('/content/heterogeneous-graph-recsys'):
    !git clone -b $BRANCH $REPO
%cd /content/heterogeneous-graph-recsys
# QUAN TRONG: cac cell sau (cau hinh) GHI DE config/training.yaml + checkpoint_manager.py
# -> working tree ban -> 'git pull' se that bai am tham va Colab ket lai code CU.
# Vi vay ep dong bo cung ve dung origin/main (reset --hard chi dung file da-track,
# giu nguyen file untracked nhu /content/data va checkpoints).
!git fetch origin $BRANCH -q && git reset --hard origin/$BRANCH && git log --oneline -1

# --- Kiem tra repo da co code cold-start BAN MOI (normalize + scale nho) chua ---
cfg_txt   = open('config/training.yaml').read()
model_txt = open('src/model/bpatmp.py').read()
eval_txt  = open('scripts/evaluate.py').read()
rt_txt    = open('scripts/run_training.py').read()
missing = []
if 'cold_start' not in cfg_txt:
    missing.append("config/training.yaml thieu block 'cold_start'")
if 'content_item' not in model_txt:
    missing.append("src/model/bpatmp.py thieu content-embedding")
if 'F.normalize(self.input_proj' not in model_txt:
    missing.append("src/model/bpatmp.py CHUA co fix normalize content (ban cu bi collapse) -> push lai!")
if 'num_nodes_dict=node_counts' not in rt_txt:
    missing.append("scripts/run_training.py CHUA co fix num_nodes_dict (sampler OOB) -> push lai!")
if 'train_mask_test_purchase_only' not in eval_txt:
    missing.append("scripts/evaluate.py chua co duong eval test trainval")
assert not missing, (
    "Repo tren GitHub CHUA co ban FIX moi nhat. Hay commit & push nhanh main roi chay lai:\n  "
    + "\n  ".join(missing)
)
print('OK: repo da co ban fix moi nhat (normalize content + num_nodes_dict + eval test).')

## 4. Tải dataset HuggingFace -> `/content/data`
[`nguyenmaiductrong/rees46-full-temporal`](https://huggingface.co/datasets/nguyenmaiductrong/rees46-full-temporal) - ~3.6 GB artifact đã tiền xử lý (feed thẳng vào training, không cần Spark).

In [4]:
import os, time
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download
t0 = time.time()
snapshot_download(repo_id='nguyenmaiductrong/rees46-full-temporal', repo_type='dataset',
                  local_dir='/content/data', max_workers=8)
print(f'Tải xong trong {time.time()-t0:.0f}s')
import json; print('node_counts:', json.load(open('/content/data/node_counts.json')))
!ls /content/data | head

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 47 files:   0%|          | 0/47 [00:00<?, ?it/s]

Tải xong trong 1s
node_counts: {'user': 245778, 'product': 100775, 'category': 14, 'brand': 3470}
candidate_item_idx.npy
cart_train_dst.npy
cart_train_src.npy
cart_train_ts.npy
cart_trainval_dst.npy
cart_trainval_src.npy
cart_trainval_ts.npy
graph
item_is_cold.npy
node_counts.json


In [ ]:
# Đảm bảo cờ cold tồn tại (cold = chưa từng purchase trong train). Nếu HF dataset đã
# có sẵn user_is_cold.npy/item_is_cold.npy thì giữ nguyên; thiếu thì tự suy ra để
# breakdown cold/warm luôn chạy được.
import json
import numpy as np
from pathlib import Path
D = Path('/content/data')
nc = json.load(open(D / 'node_counts.json'))
pu_src = np.load(D / 'purchase_train_src.npy')
pu_dst = np.load(D / 'purchase_train_dst.npy')
if not (D / 'user_is_cold.npy').exists():
    warm_u = np.zeros(nc['user'], bool); warm_u[np.unique(pu_src)] = True
    np.save(D / 'user_is_cold.npy', ~warm_u); print('Đã suy ra user_is_cold.npy')
if not (D / 'item_is_cold.npy').exists():
    warm_i = np.zeros(nc['product'], bool); warm_i[np.unique(pu_dst)] = True
    np.save(D / 'item_is_cold.npy', ~warm_i); print('Đã suy ra item_is_cold.npy')
uic = np.load(D / 'user_is_cold.npy'); iic = np.load(D / 'item_is_cold.npy')
print('user_is_cold: %.1f%% cold (%d) | item_is_cold: %.1f%% cold (%d)' % (
    100 * uic.mean(), int(uic.sum()), 100 * iic.mean(), int(iic.sum())))

## 5. Cấu hình W&B (project = `recsys-graph`) + đăng nhập

In [ ]:
import os
from pathlib import Path
import yaml

P = 'config/training.yaml'
cfg = yaml.safe_load(open(P))

WANDB_ENTITY = 'nguyenmaiductrong37-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng'
cfg['wandb']['entity'] = WANDB_ENTITY
cfg['wandb']['enabled'] = True
# project / artifact_name / run_name giữ theo config mới (bpatmp-recsys / bpatmp-final-l4)
cfg['data']['data_dir'] = '/content/data'
cfg['data']['struct_dir'] = '/content/data/node_mappings'
cfg['training']['progress_bar'] = False
cfg['training']['quiet_checkpoint_logs'] = True

# Cold-start: đảm bảo block tồn tại (đã có sẵn trong repo; default phòng hờ).
cs = cfg.setdefault('cold_start', {})
cs.setdefault('content_item', True)
cs.setdefault('content_scale_init', 0.05)
cs.setdefault('p_id', 0.15)
cs.setdefault('p_hist', 0.2)

os.environ['WANDB_ENTITY'] = WANDB_ENTITY
os.environ['WANDB_PROJECT'] = cfg['wandb']['project']
os.environ['WANDB_SILENT'] = 'true'
yaml.safe_dump(cfg, open(P, 'w'), sort_keys=False, allow_unicode=True)

# Checkpoint artifact ~1-1.5GB: tăng thời gian verify cloud và tránh artifact ref entity=None.
cm_path = Path('src/training/checkpoint_manager.py')
txt = cm_path.read_text()
txt = txt.replace('verify_timeout_secs: int = 300,', 'verify_timeout_secs: int = 3600,')
txt = txt.replace('verify_poll_secs: int = 30,', 'verify_poll_secs: int = 60,')
txt = txt.replace('wandb.Api(timeout=60).artifact', 'wandb.Api(timeout=180).artifact')
txt = txt.replace(
      " artifact.add_file(str(ckpt_path))",
      " artifact.add_file(str(ckpt_path), policy=\"immutable\", skip_cache=True)",
  )
old = """        self._save_run_id(self.run.id)\n        logger.info(\"W&B run ready: id=%s\", self.run.id)\n"""
new = """        self.project = self.run.project or self.project\n        self.entity = self.run.entity or self.entity\n        if not self.entity:\n            raise RuntimeError(\n                \"W&B entity is empty after wandb.init(). Set wandb.entity in config.\"\n            )\n        self._save_run_id(self.run.id)\n        logger.info(\n            \"W&B run ready: entity=%s project=%s name=%s id=%s\",\n            self.entity, self.project, self.run.name, self.run.id,\n        )\n"""
if old in txt:
    txt = txt.replace(old, new)
txt = txt.replace(
    "    def _cleanup_old_checkpoints(self, keep: Path) -> None:\n        for pt in list(self.local_dir.glob(\"*.pt\")) + list(self.local_dir.glob(\"*.pth\")):\n            if pt.resolve() != keep.resolve():",
    "    def _cleanup_old_checkpoints(self, keep: Path) -> None:\n        for pt in list(self.local_dir.glob(\"*.pt\")) + list(self.local_dir.glob(\"*.pth\")):\n            if pt.name == \"best.pt\":\n                continue\n            if pt.resolve() != keep.resolve():",
)
cm_path.write_text(txt)

print('wandb: entity =', cfg['wandb']['entity'], '| project =', cfg['wandb']['project'],
      '| artifact =', cfg['wandb']['artifact_name'], '| data_dir =', cfg['data']['data_dir'])
print('cold_start:', cfg['cold_start'])
print('model:', cfg['model'])
print('training: epochs=%s batch=%s eval_every=%s eval_subsample=%s primary=%s device=%s save_dir=%s' % (
    cfg['training']['epochs'], cfg['training']['batch_size'], cfg['training'].get('eval_every'),
    cfg['training'].get('eval_subsample'), cfg['evaluation'].get('primary_metric'),
    cfg['training']['device'], cfg['training']['save_dir']))

import wandb
try:
    from google.colab import userdata
    wandb.login(key=userdata.get('WANDB_API_KEY'))
except Exception as e:
    print('Không thấy secret WANDB_API_KEY, đăng nhập thủ công:', e)
    wandb.login()

## 6. Training (đã bao gồm tự đánh giá VAL + cold/warm)
`scripts/run_training.py` mỗi epoch: chạy `eval_epoch` trên **val**, log HR/NDCG@{1,5,10,20,50} + breakdown **`cold_user/*`** và **`warm_user/*`** (chia theo `user_is_cold.npy`), chọn `best.pt` theo **NDCG@20** tổng, early stopping (`patience`). Kết thúc chạy **full-val trên toàn bộ user** với `best.pt` kèm cold/warm.

Cold-start được học ngay trong vòng train: item embedding nối đất bằng **category + brand**, **ID-dropout** (`p_id`) và **history-dropout** (`p_hist`) mô phỏng item/user thưa.

40 epoch trên dữ liệu 245.778 user / 100.775 item (full-ranking) - tùy A100 có thể mất vài giờ. Muốn thử nhanh: giảm `epochs` ở cell 5.

In [ ]:
!python scripts/run_training.py --config config/training.yaml

## 7. Đánh giá TEST trên checkpoint tốt nhất
Training (cell 6) đã đánh giá **val** rồi. Phần này chạy split **test** - giao thức rolling-temporal (graph train+val, mask train+val) mà training **không bao giờ đụng tới**. Đây là số liệu cuối cùng.

(Cell val bên dưới chỉ để xác nhận lại, có thể bỏ qua.)

In [9]:
# Tải TẤT CẢ version checkpoint từ W&B artifact (mỗi epoch = 1 version)
import os, shutil, yaml
from pathlib import Path
import wandb

cfg_w    = yaml.safe_load(open('config/training.yaml'))['wandb']
ENTITY   = cfg_w.get('entity') or os.environ.get('WANDB_ENTITY')
PROJECT  = cfg_w.get('project', 'recsys-graph')
ARTIFACT = cfg_w.get('artifact_name')          # vd:
CKPT_DIR = Path('/content/all_checkpoints')
shutil.rmtree(CKPT_DIR, ignore_errors=True); CKPT_DIR.mkdir(parents=True, exist_ok=True)

path = f"{ENTITY}/{PROJECT}/{ARTIFACT}"
print('Artifact path:', path)
api = wandb.Api()
try:
    versions = list(api.artifacts("model", path))           # API moi
except Exception:
    versions = list(api.artifact_versions("model", path))   # fallback API cu

def _vidx(v):
    try:    return int(str(v.version).lstrip('v'))
    except Exception: return 0
versions = sorted(versions, key=_vidx)

for v in versions:
    label = next((a for a in (v.aliases or []) if str(a).startswith('epoch-')), str(v.version))
    v.download(root=str(CKPT_DIR / label))
    print(f'OK {label:14s} <- {v.version}')
print(f'\nDa tai {len(versions)} checkpoint vao {CKPT_DIR}')

Artifact path: nguyenmaiductrong37-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng/recsys-graph/recsys-graph-v5-l2
OK epoch-000      <- v0
OK epoch-001      <- v1
OK epoch-002      <- v2
OK epoch-003      <- v3
OK epoch-004      <- v4
OK epoch-005      <- v5
OK epoch-006      <- v6
OK epoch-007      <- v7
OK epoch-008      <- v8
OK epoch-009      <- v9
OK epoch-010      <- v10
OK epoch-011      <- v11
OK epoch-012      <- v12
OK epoch-013      <- v13
OK epoch-014      <- v14
OK epoch-015      <- v15
OK epoch-016      <- v16
OK epoch-017      <- v17
OK epoch-018      <- v18
OK epoch-019      <- v19
OK epoch-020      <- v20
OK epoch-021      <- v21
OK epoch-022      <- v22
OK epoch-023      <- v23
OK epoch-024      <- v24
OK epoch-025      <- v25
OK epoch-026      <- v26
OK epoch-027      <- v27
OK epoch-028      <- v28
OK epoch-029      <- v29
OK epoch-030      <- v30
OK epoch-031      <- v31
OK epoch-032      <- v32

Da tai 33 checkpoint vao /content/all_checkpoints


In [ ]:
# Evaluate lần lượt từng checkpoint trên split TEST, gom kết quả thành bảng
import json, subprocess
from pathlib import Path
import pandas as pd

rows  = []
ckpts = sorted(Path('/content/all_checkpoints').rglob('*.pt'), key=lambda p: p.parent.name)
print(f'Co {len(ckpts)} checkpoint can danh gia\n')
for ckpt in ckpts:
    tag = ckpt.parent.name
    print(f'===== {tag}  ({ckpt.name}) =====')
    res = subprocess.run(
        ['python', '-u', 'scripts/evaluate.py', '--split', 'test', '--checkpoint', str(ckpt)],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        print('  LOI:', res.stderr[-1500:]); continue
    # evaluate.py ghi eval_test.json canh checkpoint -> doc cho chac
    jf = ckpt.parent / 'eval_test.json'
    if not jf.exists():
        print('  Khong thay eval_test.json'); print(res.stdout[-800:]); continue
    d = json.loads(jf.read_text())
    m = dict(d.get('metrics', {})); m['checkpoint'] = tag; m['epoch'] = d.get('epoch')
    rows.append(m)
    print('  NDCG@20=%.4f | NDCG@10=%.4f | HR@10=%.4f | warm NDCG@10=%.4f | cold NDCG@10=%.4f' % (
        m.get('NDCG@20', float('nan')), m.get('NDCG@10', float('nan')), m.get('HR@10', float('nan')),
        m.get('warm_user/NDCG@10', float('nan')), m.get('cold_user/NDCG@10', float('nan'))))

df = pd.DataFrame(rows)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)
show = ['checkpoint','epoch','HR@10','NDCG@10','HR@20','NDCG@20',
        'warm_user/HR@10','warm_user/NDCG@10','cold_user/HR@10','cold_user/NDCG@10']
show = [c for c in show if c in df.columns]
print('\n========== TONG HOP (test) ==========')
print(df[show].to_string(index=False) if not df.empty else 'Khong co ket qua.')
df.to_csv('/content/eval_all_checkpoints.csv', index=False)
print('\nDa luu bang day du: /content/eval_all_checkpoints.csv')
# Primary metric cua du an = NDCG@20 (full-ranking)
if not df.empty and 'NDCG@20' in df.columns:
    b = df.loc[df['NDCG@20'].idxmax()]
    print('-> Best NDCG@20: %s (epoch %s)  NDCG@20=%.4f  NDCG@10=%.4f  HR@10=%.4f  | warm NDCG@10=%.4f  cold NDCG@10=%.4f' % (
        b['checkpoint'], b.get('epoch'), b['NDCG@20'], b.get('NDCG@10', float('nan')), b.get('HR@10', float('nan')),
        b.get('warm_user/NDCG@10', float('nan')), b.get('cold_user/NDCG@10', float('nan'))))

## Ghi chú
- Metric chính: **NDCG@20** (full-ranking trên toàn bộ item).
- Checkpoint & metric lưu cả local (`save_dir`) lẫn W&B artifact trong project `recsys-graph`.
- Nếu A100 hết RAM: giảm `training.batch_size` / `eval_batch_size` ở cell 5.
- Nếu phiên Colab ngắt giữa chừng: chạy lại cell Training - code tự resume từ checkpoint mới nhất trong `save_dir`.